# MODULE B - Data Engineering
#### Objective: The objective of Module B is to design, implement, and validate a reproducible, scalable, and auditable data engineering pipeline that transforms raw IBM AMLSim transaction data into clean, structured, and analysis-ready datasets for coordinated fraud ring detection and investigation.The pipeline must reliably process approximately 100,000 accounts, 12.48 million transactions, and 17,052 alert records, while preserving transaction integrity, account relationships, temporal ordering, and the original synthetic fraud labels.The module will focus on five engineering components:
#### B1. Raw data ingestion 
#### B2. Schema validation 
#### B3. Data cleaning and quarantine 
#### B4. SQL analytical storage 
#### B5. Reproducible pipeline


# B1 -Raw Data Ingestion
#### Objective: Develop a memory-efficient ingestion process for reading large CSV files, preserving source data, standardizing input handling, and recording ingestion metadata.

In [25]:
#B1.1 Recursive raw dataset inventory
from pathlib import Path
from datetime import datetime, timezone
import csv
import stat
DATASET_ROOT = Path(r"C:\Users\maiti\CAREER_HQ\09_PROJECTS_PORTFOLIO\Coordinated Fraud Ring Detection and Investigation\dataset\raw\100Kvertices-10Medges")
OUTPUT_PATH = Path(r"C:\Users\maiti\CAREER_HQ\09_PROJECTS_PORTFOLIO\Coordinated Fraud Ring Detection and Investigation\reports\dataset_manifest.csv")

FIELDNAMES = [
    "relative_path",
    "file_name",
    "extension",
    "size_bytes",
    "last_modified_utc",
]
def validate_paths(root: Path, output: Path) -> None:
    root=root.resolve()
    output=output.resolve()
    if not root.exists():
        raise FileNotFoundError(f'Dataset root not exist: {root}')
    if not root.is_dir():
        raise NotADirectoryError(f'Dataset root is not a directory : {root}')
    if output.suffix.lower() != ".csv" :
        raise ValueError("Output file must have a .csv extension")
    try:
       output.relative_to(root)
    except ValueError:
        pass
    else:
        raise ValueError("Output file cannot be inside the dataset root")
import os
from pathlib import Path

def discover_files(root: Path) -> list[Path]:
    files = []

    def handle_error(error):
        raise OSError(
            f"Failed to enumerate directory: {error.filename}"
        ) from error

    for dirpath, dirnames, filenames in os.walk(
        root,
        topdown=True,
        onerror=handle_error,
        followlinks=False
    ):
        current_dir = Path(dirpath)

        for filename in filenames:
            file_path = current_dir / filename

            try:
                file_stat = file_path.lstat()
            except OSError as e:
                raise OSError(f"Failed to retrieve metadata for {file_path}") from e

            if stat.S_ISLNK(file_stat.st_mode):
                continue

            if stat.S_ISREG(file_stat.st_mode):
                files.append(file_path)

    files.sort(
        key=lambda file: file.relative_to(root).as_posix()
    )

    return files
    

    
def extract_metadata(file_path: Path, root: Path) -> dict:
    #get file metadata once
    try:
        file_stats=file_path.stat()
    except OSError as e:
        raise OSError(f"failed to retrieve metadata for {file_path}") from e
    #POSIX - style relative path
    relative_path=file_path.relative_to(root).as_posix()
    #file name and extension
    file_name=file_path.name
    extension=file_path.suffix
    #file size in bytes
    size_bytes=file_stats.st_size
    # UTC ISO 8601 timestamp
    last_modified_utc = datetime.fromtimestamp(
        file_stats.st_mtime,
        tz=timezone.utc
    ).isoformat(timespec="microseconds")

    # Return metadata dictionary
    return {
        "relative_path": relative_path,
        "file_name": file_name,
        "extension": extension,
        "size_bytes": size_bytes,
        "last_modified_utc": last_modified_utc
    }
def build_manifest(root: Path) -> list[dict]:
    rows = []

    # Iterate through all discovered files
    for file_path in discover_files(root):
        rows.append(extract_metadata(file_path, root))

    return rows
def write_manifest(rows: list[dict], output: Path) -> None:
    # create parent directory if needed
    output.parent.mkdir(parents=True, exist_ok=True)
    with output.open("w",encoding="utf-8",newline="") as file:
        #create CSV writer
        writer=csv.DictWriter(file,fieldnames=FIELDNAMES,lineterminator="\n")
        #write header
        writer.writeheader()
        #write all rows
        writer.writerows(rows)


    

In [26]:
#main execution cell of B1.1
validate_paths(DATASET_ROOT, OUTPUT_PATH)

rows = build_manifest(DATASET_ROOT)

write_manifest(rows, OUTPUT_PATH)

print(f"Manifest created: {OUTPUT_PATH}")
print(f"Files inventoried: {len(rows)}")

Manifest created: C:\Users\maiti\CAREER_HQ\09_PROJECTS_PORTFOLIO\Coordinated Fraud Ring Detection and Investigation\reports\dataset_manifest.csv
Files inventoried: 14


In [27]:
#repeatibility testing of B1.1
manifest1 = build_manifest(DATASET_ROOT)
manifest2 = build_manifest(DATASET_ROOT)

assert manifest1 == manifest2

print("Repeatability test passed")


Repeatability test passed


#### Failure Scenario Testing of section B1.1

In [28]:
#test 1 : missing dataset
try:
    validate_paths(
        Path("missing_folder"),
        OUTPUT_PATH
    )
except FileNotFoundError:
    print("PASS")

PASS


In [29]:
#test 2 : Output not csv
try:
    validate_paths(
        DATASET_ROOT,
        Path("report.txt")
    )
except ValueError:
    print("PASS")

PASS


In [30]:
#test 3 : Ouput inside dataset
try:
    validate_paths(
        DATASET_ROOT,
        DATASET_ROOT / "manifest.csv"
    )
except ValueError:
    print("PASS")

PASS


In [31]:
#test 4 : metadata failure
try:
    extract_metadata(
        Path("does_not_exist.csv"),
        DATASET_ROOT
    )
except OSError:
    print("PASS")

PASS


In [33]:
#test 4 : directory-scan failure
import os

original_walk = os.walk

def failing_walk(*args, **kwargs):
    kwargs["onerror"](
        OSError("Simulated directory scan failure")
    )
    return []

os.walk = failing_walk

try:
    discover_files(DATASET_ROOT)
except OSError:
    print("PASS: Directory-scan failure detected")
else:
    raise AssertionError(
        "FAIL: Expected OSError was not raised"
    )
finally:
    os.walk = original_walk

PASS: Directory-scan failure detected


In [36]:
#test - 5:discovery stage metadata failure
# TEST: Discovery-stage metadata failure

from pathlib import Path

original_lstat = Path.lstat

def failing_lstat(self):
    raise OSError(
        "Simulated metadata access failure"
    )

Path.lstat = failing_lstat

try:
    discover_files(DATASET_ROOT)
except OSError:
    print("PASS: Metadata failure detected")
else:
    raise AssertionError(
        "FAIL: Expected OSError was not raised"
    )
finally:
    Path.lstat = original_lstat



PASS: Metadata failure detected


In [32]:
#B1.2 - Header, schema and sample inspection